# RSNA Knee Abnormality Detection
### DINOv2 Baseline with LLM Labels and Geometric Preprocessing

In [ ]:
# CELL 1: Imports and Environment Setup
import os
import sys
import gc
import math
import numpy as np
import pandas as pd
import pydicom
from pydicom.pixel_data_handlers.util import apply_voi_lut
import cv2
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.amp

class CFG:
    seed = 42
    base_dir = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
    train_csv = os.path.join(base_dir, "train.csv")
    train_series_csv = os.path.join(base_dir, "train_series.csv")
    train_images_dir = os.path.join(base_dir, "train_series")
    
    llm_labels_dir = "/kaggle/input/datasets/pilkwang/rsna-knee-llm-labels"
    weights_dir = "/kaggle/input/datasets/pilkwang/rsna-knee-weights"
    
    crop_mm = 130.0 
    image_size = 336
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    n_folds = 5
    fold_to_train = 0

torch.manual_seed(CFG.seed)
np.random.seed(CFG.seed)

train_df = pd.read_csv(CFG.train_csv)
train_series_df = pd.read_csv(CFG.train_series_csv)
print(f"Loaded train.csv: {train_df.shape} | train_series.csv: {train_series_df.shape}\n")


In [ ]:
# CELL 2: Exploratory Data Analysis & Label Integration
import seaborn as sns
import matplotlib.pyplot as plt

llm_csv = os.path.join(CFG.llm_labels_dir, "train.csv") 
if not os.path.exists(llm_csv):
    llm_csv = os.path.join(CFG.llm_labels_dir, "train_llm_labels.csv")

print(f"Loading LLM Labels from: {llm_csv}")
if os.path.exists(llm_csv):
    llm_df = pd.read_csv(llm_csv)

    target_cols = [
        "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", 
        "Medial OA", "Lateral OA", "PF OA", "Effusion", 
        "Synovitis", "Baker's", "Contusion", "Fracture"
    ]

    train_df.set_index("StudyInstanceUID", inplace=True)
    llm_df.set_index("StudyInstanceUID", inplace=True)
    train_df.update(llm_df)
    train_df.reset_index(inplace=True)

    train_df = train_df.dropna(subset=target_cols).reset_index(drop=True)
    print(f"Total training studies available after LLM merge: {len(train_df)}")

    prevalence = train_df[target_cols].mean().sort_values(ascending=False) * 100
    plt.figure(figsize=(12, 5))
    sns.barplot(x=prevalence.values, y=prevalence.index, palette="viridis")
    plt.title("Prevalence of Knee Abnormalities in Training Set (%)")
    plt.xlabel("Percentage")
    plt.show()
else:
    print(f"WARNING: LLM labels not found at {llm_csv}. Please attach the dataset!")


In [ ]:
# CELL 3: DICOM Geometry Engine
def get_slice_position(dcm):
    if hasattr(dcm, "ImageOrientationPatient") and hasattr(dcm, "ImagePositionPatient"):
        try:
            pos = np.array([float(x) for x in dcm.ImagePositionPatient])
            ori = np.array([float(x) for x in dcm.ImageOrientationPatient])
            normal = np.cross(ori[0:3], ori[3:6])
            return np.dot(pos, normal)
        except:
            pass
    if hasattr(dcm, "SliceLocation"): return float(dcm.SliceLocation)
    if hasattr(dcm, "InstanceNumber"): return float(dcm.InstanceNumber)
    return 0.0

def sort_dicom_files(dicom_paths):
    metadata = []
    for path in dicom_paths:
        try:
            dcm = pydicom.dcmread(path, stop_before_pixels=True)
            metadata.append({"path": path, "pos": get_slice_position(dcm), "instance": getattr(dcm, "InstanceNumber", 0)})
        except:
            continue
    metadata.sort(key=lambda x: (x["pos"], x["instance"], x["path"]))
    return [x["path"] for x in metadata]

def read_and_crop_dicom(path, crop_mm=130.0, target_size=336):
    try:
        dcm = pydicom.dcmread(path)
        img = apply_voi_lut(dcm.pixel_array, dcm)
        if dcm.PhotometricInterpretation == "MONOCHROME1": img = np.amax(img) - img
        img = img - np.min(img)
        img = img / (np.max(img) + 1e-6)
        img = (img * 255).astype(np.uint8)
        
        spacing_y, spacing_x = 1.0, 1.0
        if hasattr(dcm, "PixelSpacing"):
            try: spacing_y, spacing_x = [float(s) for s in dcm.PixelSpacing]
            except: pass
        
        h, w = img.shape
        crop_h, crop_w = min(int(crop_mm / spacing_y), h), min(int(crop_mm / spacing_x), w)
        start_y, start_x = (h - crop_h) // 2, (w - crop_w) // 2
        
        cropped_img = img[start_y:start_y+crop_h, start_x:start_x+crop_w]
        return cv2.resize(cropped_img, (target_size, target_size), interpolation=cv2.INTER_CUBIC)
    except:
        return np.zeros((target_size, target_size), dtype=np.uint8)


In [ ]:
# CELL 4: 3-Slice RGB Triplet Dataset
import albumentations as A

class RSNAKneeDataset(Dataset):
    def __init__(self, df, series_df, is_train=True):
        self.df, self.series_df, self.is_train = df, series_df, is_train
        self.target_cols = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]
        self.transform = A.Compose([A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        study_id = row["StudyInstanceUID"]
        study_series = self.series_df[self.series_df["StudyInstanceUID"] == study_id]
        series_dir = os.path.join(CFG.train_images_dir, study_id, study_series.iloc[0]["SeriesInstanceUID"])
        
        sorted_files = sort_dicom_files([os.path.join(series_dir, f) for f in os.listdir(series_dir) if f.endswith(".dcm")])
        
        num_triplets = 3
        images = []
        if len(sorted_files) >= 3:
            valid_files = sorted_files[int(len(sorted_files)*0.2):int(len(sorted_files)*0.8)]
            if len(valid_files) < 3: valid_files = sorted_files
            anchors = np.linspace(0, len(valid_files)-1, num_triplets, dtype=int)
            
            for anchor in anchors:
                img_R = read_and_crop_dicom(valid_files[max(0, anchor-1)], CFG.crop_mm, CFG.image_size)
                img_G = read_and_crop_dicom(valid_files[anchor], CFG.crop_mm, CFG.image_size)
                img_B = read_and_crop_dicom(valid_files[min(len(valid_files)-1, anchor+1)], CFG.crop_mm, CFG.image_size)
                
                rgb_img = self.transform(image=np.stack([img_R, img_G, img_B], axis=-1))["image"]
                images.append(torch.tensor(rgb_img, dtype=torch.float32).permute(2, 0, 1))
                
        while len(images) < num_triplets: images.append(torch.zeros((3, CFG.image_size, CFG.image_size)))
        
        images_tensor = torch.stack(images)
        if self.is_train:
            return images_tensor, torch.tensor(row[self.target_cols].values.astype(np.float32))
        return images_tensor


In [ ]:
# CELL 5: DINOv2 Model
import timm

class DINOv2KneeModel(nn.Module):
    def __init__(self, model_name="vit_small_patch14_reg4_dinov2.lvd142m", num_classes=12):
        super().__init__()
        self.encoder = timm.create_model(model_name, pretrained=True, num_classes=0, dynamic_img_size=True)
        self.head = nn.Sequential(nn.Dropout(0.2), nn.Linear(self.encoder.num_features, num_classes))

    def forward(self, x):
        batch_size, num_triplets, c, h, w = x.shape
        features = self.encoder(x.view(-1, c, h, w))
        logits, _ = torch.max(self.head(features).view(batch_size, num_triplets, -1), dim=1) 
        return logits

model = DINOv2KneeModel().to(CFG.device)
print("Model initialized successfully!")


In [ ]:
# CELL 6: Training and Validation Functions
from sklearn.metrics import roc_auc_score

def train_epoch(model, dataloader, criterion, optimizer, scaler, device):
    model.train()
    running_loss = 0.0
    pbar = tqdm(dataloader, desc="Training")
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        with torch.amp.autocast("cuda"):
            loss = criterion(model(images), labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item() * images.size(0)
        pbar.set_postfix({"loss": f"{loss.item():.4f}"})
    return running_loss / len(dataloader.dataset)

@torch.no_grad()
def valid_epoch(model, dataloader, criterion, device):
    model.eval()
    running_loss, all_targets, all_preds = 0.0, [], []
    for images, labels in tqdm(dataloader, desc="Validation"):
        images, labels = images.to(device), labels.to(device)
        with torch.amp.autocast("cuda"):
            logits = model(images)
            loss = criterion(logits, labels)
        running_loss += loss.item() * images.size(0)
        all_targets.append(labels.cpu().numpy())
        all_preds.append(torch.sigmoid(logits).cpu().numpy())
        
    all_targets, all_preds = np.vstack(all_targets), np.vstack(all_preds)
    auc_scores = [roc_auc_score(all_targets[:, i], all_preds[:, i]) for i in range(12) if len(np.unique(all_targets[:, i])) > 1]
    return running_loss / len(dataloader.dataset), np.mean(auc_scores) if auc_scores else 0.0


In [ ]:
# CELL 7: Stratified 5-Fold DataLoaders
from sklearn.model_selection import KFold

kf = KFold(n_splits=CFG.n_folds, shuffle=True, random_state=CFG.seed)
train_df["fold"] = -1
for fold, (train_idx, val_idx) in enumerate(kf.split(train_df)):
    train_df.loc[val_idx, "fold"] = fold

print(train_df["fold"].value_counts().sort_index())

train_split_df = train_df[train_df["fold"] != CFG.fold_to_train].reset_index(drop=True)
val_split_df = train_df[train_df["fold"] == CFG.fold_to_train].reset_index(drop=True)
print(f"Training on {len(train_split_df)} studies, Validating on {len(val_split_df)} studies (Fold {CFG.fold_to_train})")

BATCH_SIZE = 8
train_loader = DataLoader(RSNAKneeDataset(train_split_df, train_series_df), batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(RSNAKneeDataset(val_split_df, train_series_df), batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print("Distributed Folds & DataLoaders ready for competition scale!")


In [ ]:
# CELL 8: The Execution Loop
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

EPOCHS = 5
criterion = nn.BCEWithLogitsLoss()
optimizer = AdamW(model.parameters(), lr=2e-4, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = torch.amp.GradScaler("cuda")
best_auc = 0.0

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    train_loss = train_epoch(model, train_loader, criterion, optimizer, scaler, CFG.device)
    val_loss, val_auc = valid_epoch(model, val_loader, criterion, CFG.device)
    scheduler.step()
    print(f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val AUC: {val_auc:.4f}")
    if val_auc > best_auc:
        best_auc = val_auc
        torch.save(model.state_dict(), f"dinov2_knee_fold{CFG.fold_to_train}_best.pth")
        print("--> Saved best model!")
